In [21]:
# === Core Python ===
import os
import collections
from datetime import datetime
import datetime as dt
import json 
import glob 

# === Numerical and Data Handling ===
import numpy as np
import numpy.ma as ma
import pandas as pd
import xarray as xr
import xcdat as xcd
import xskillscore as xs

In [24]:
def read_nudge_tend_data(model_list, exp_dict, period, ttag, season,
                         var_dict, out_path):
    year = period.split("_")[0][0:4]
    for exp in model_list:
        print(f"Processing: {exp}")
        if 'NDGUV' in exp:
            case = "E3SMv2_NDGUVTQ_SRF1_tau6_3hourly"
            base_path = "/pscratch/sd/z/zhan391/seacrogs_scratch/nudge_training/post/atm/180x360_aave/clim"
        else:
            case = exp_dict[exp]['casename']
            base_path = exp_dict[exp]['path_template']

        search_path = os.path.join(base_path, ttag, f"{case}*{season}*{year}*.nc")
        file_list = glob.glob(search_path)
        if not file_list:
            raise FileNotFoundError(f"No NetCDF files found for {exp} at {search_path}")

        ds = xr.open_dataset(file_list[0])
        dsm = ds.mean(dim="time", keep_attrs=True)

        # Process and scale each variable
        var_data = {}
        for var in var_dict.keys():
            vfac = var_dict[var]['fact'] * 10800.0 if 'NDGUV' in exp else var_dict[var]['fact'] * 10800.0 * 6
            if 'lev' in dsm.dims:
                data = (dsm[var] * vfac).mean(dim='lev', skipna=True)
            else:
                data = dsm[var] * vfac
            data.attrs['units'] =  var_dict[var]['unit']
            data.attrs['name'] =  var_dict[var]['name']
            var_data[var] = data
        # Combine into Dataset and save
        ds_proc = xr.Dataset(var_data, coords={'lat': dsm.lat, 'lon': dsm.lon})
        ds_proc.attrs['description'] = f"Processed nudge tendency data for {season} {period}"
        ds_proc.attrs['note'] = "Time-averaged, vertically averaged (if applicable), and scaled to per 3hour"

        out_dir = os.path.join(out_path, ttag)
        os.makedirs(out_dir, exist_ok=True)
        out_file = os.path.join(out_dir, f"{exp}_nudge_tend_{season}.nc")
        if os.path.exists(out_file):
            os.remove(out_file)
        ds_proc.to_netcdf(out_file)
        print(f"Saved processed data to: {out_file}")
        
    return 

In [25]:
if __name__ == "__main__":
    # === Define Paths ===
    top_path  = "/pscratch/sd/z/zhan391/SEACROGS_project"
    work_path = f"{top_path}/paper_material/method_paper"
    out_path  = f"{work_path}/fig_data"
    fig_path  = f"./figures"
    os.makedirs(fig_path, exist_ok=True)
    os.makedirs(out_path, exist_ok=True)

    # === Time Info and Reference ===
    period  = "201201_201612"
    ttag    = "5yr"
    seasons = ["ANN", "DJF", "JJA", "SON", "MAM"]
    
    # === Variable settings ===
    var_dict = {
        "Nudge_U": {
            'name': "UTEND",
            'unit': 'm s$^{-1}$ 3hr$^{-1}$',
            'fact': 1.0
        },
        "Nudge_V": {
            'name': "VTEND",
            'unit': 'm s$^{-1}$ 3hr$^{-1}$',
            'fact': 1.0
        },
        "Nudge_T": {
            'name': "TTEND",
            'unit': 'K 3hr$^{-1}$',
            'fact': 1.0
        },
        "Nudge_Q": {
            'name': "QTEND",
            'unit': 'g kg$^{-1}$ 3hr$^{-1}$',
            'fact': 1e3
        }
    }
    
    # === Load experiment dictionary ===
    json_file = f"{work_path}/experiment_info/e3sm_experiments.json"
    with open(json_file, "r") as f:
        exp_dict = json.load(f)

    # === Define experiment group ===
    ref_key = "NDGUVTQ"
    group = "online_ML_IMT_cut100"
    model_dict = {
        "NDGUVTQ":              "Nudge",
        "UNET_IMT_cut100":      "UNet",
        "UNETMP_IMT_cut100":    "UNetMP",
        "IUNET_IMT_cut100":     "IUNet",
        "MnM_IMT_cut100":       "MnM"
    }
    model_list = list(model_dict.keys())

    # === Process and Save Data for Each Season ===
    for season in seasons:
        print(f"\n--- Processing {season} ---")
        read_nudge_tend_data(
            model_list=model_list,
            exp_dict=exp_dict,
            period=period,
            ttag=ttag,
            season=season,
            var_dict=var_dict,
            out_path=out_path
        )


--- Processing ANN ---
Processing: NDGUVTQ
Saved processed data to: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/5yr/NDGUVTQ_nudge_tend_ANN.nc
Processing: UNET_IMT_cut100
Saved processed data to: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/5yr/UNET_IMT_cut100_nudge_tend_ANN.nc
Processing: UNETMP_IMT_cut100
Saved processed data to: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/5yr/UNETMP_IMT_cut100_nudge_tend_ANN.nc
Processing: IUNET_IMT_cut100
Saved processed data to: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/5yr/IUNET_IMT_cut100_nudge_tend_ANN.nc
Processing: MnM_IMT_cut100
Saved processed data to: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/5yr/MnM_IMT_cut100_nudge_tend_ANN.nc

--- Processing DJF ---
Processing: NDGUVTQ
Saved processed data to: /pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/5y